# vn-traffic-law-rag · dựng index + ablation

Chạy trên Kaggle hoặc Colab, **CPU là đủ** (corpus chỉ vài nghìn chunk), bật Internet.
Kaggle: Settings → Accelerator → None (CPU). Colab: runtime CPU mặc định là được.
Input: trên Kaggle là dataset `vn-traffic-law-rag-code` (code + `data/` của repo); trên Colab notebook tự clone
repo từ GitHub và lưu kết quả vào Google Drive (`MyDrive/ai-portfolio/vn-traffic-law-rag`).
Secrets (Kaggle: Add-ons → Secrets; Colab: biểu tượng chìa khoá bên trái, bật Notebook access):
`GEMINI_API_KEY`, `GROQ_API_KEY` (sinh câu hỏi eval), `HF_TOKEN` (đẩy index/model lên Hub).

Các bước:
1. Dựng chunks từ snapshot vbpl
2. Embed bằng 2 model: `multilingual-e5-small` (không cần tách từ) và `bkai vietnamese-bi-encoder` (cần pyvi)
3. Quantize e5-small sang ONNX int8 (bản `avx512_vnni` trên HF không chạy được trên CPU AMD Zen 3 của mình)
4. Sinh câu hỏi ứng viên nếu chưa có bộ eval đã duyệt
5. Ablation: BM25 / dense / hybrid / +rerank / +lọc hiệu lực / +loại xe / +tham chiếu
6. Đẩy index + model ONNX lên HF Hub để laptop và HF Space tải về

In [ ]:
import glob, json, os, shutil, sys, time

IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    SRC = glob.glob("/kaggle/input/**/vnlaw_rag/__init__.py", recursive=True)[0].rsplit("/vnlaw_rag", 1)[0]
    REPO = SRC.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vn-traffic-law-rag" if IN_COLAB else os.path.abspath("vn-traffic-law-rag")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vn-traffic-law-rag {REPO}
    SRC = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vn-traffic-law-rag" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC
shutil.copytree(f"{REPO}/data", "data", dirs_exist_ok=True)
!pip install -q sentence-transformers pyvi onnxruntime onnx "huggingface_hub>=0.24"


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)


for k in ("GEMINI_API_KEY", "GROQ_API_KEY", "HF_TOKEN"):
    v = secret(k)
    if v:
        os.environ[k] = v
    else:
        print("thiếu secret", k)
HF_REPO = "DuongCodeAI/vn-traffic-law-rag-index"

In [ ]:
!python -m vnlaw_rag.build_corpus
!wc -l data/processed/chunks.jsonl

In [ ]:
from vnlaw_rag.retriever import load_chunks

chunks = load_chunks("data/processed/chunks.jsonl")
lens = sorted(len(c["text"]) for c in chunks)
print(f"{len(chunks)} chunk, độ dài ký tự p50={lens[len(lens)//2]} p95={lens[int(len(lens)*.95)]} max={lens[-1]}")
print(chunks[100]["citation"]); print(chunks[100]["text"])

## Embed bằng sentence-transformers (2 model)

In [ ]:
import numpy as np
from vnlaw_rag.dense import PRESETS, DenseIndex, Embedder

texts = [c["text"] for c in chunks]
for name in ("e5-small", "bkai"):
    cfg = PRESETS[name]
    t0 = time.time()
    emb = Embedder(cfg)
    vec = emb.encode_passages(texts, batch_size=32)
    DenseIndex(vec).save(f"data/processed/dense_{name}", cfg)
    print(name, vec.shape, f"{time.time() - t0:.0f}s")

## e5-small → ONNX int8 (cho laptop / HF Space không có torch)
Lấy `onnx/model.onnx` (fp32) có sẵn trong repo của intfloat rồi quantize dynamic sang uint8.
Sau đó embed lại corpus bằng CHÍNH bản int8 để vector câu hỏi và vector đoạn luật cùng một model.

In [ ]:
from huggingface_hub import hf_hub_download
from onnxruntime.quantization import QuantType, quantize_dynamic

os.makedirs("models/embedder", exist_ok=True)
fp32 = hf_hub_download("intfloat/multilingual-e5-small", "onnx/model.onnx")
for f in ("tokenizer.json", "config.json"):
    shutil.copy(hf_hub_download("intfloat/multilingual-e5-small", f), f"models/embedder/{f}")
quantize_dynamic(fp32, "models/embedder/model.int8.onnx", weight_type=QuantType.QUInt8)
!ls -la models/embedder

from dataclasses import replace
from vnlaw_rag.dense import EmbedConfig

cfg_onnx = replace(PRESETS["e5-small"], model_name="models/embedder", backend="onnx", onnx_file="model.int8.onnx")
t0 = time.time()
vec8 = Embedder(cfg_onnx).encode_passages(texts, batch_size=16)
DenseIndex(vec8).save("data/processed/dense", PRESETS["e5-small"])
vec32 = np.load("data/processed/dense_e5-small/vectors.npy")
print(f"int8 {time.time() - t0:.0f}s, cosine(int8, fp32) trung bình = {(vec8 * vec32).sum(1).mean():.4f}")

## Reranker: mmarco-mMiniLMv2 (đa ngữ, có bản ONNX quint8_avx2 ~119MB)

In [ ]:
RR = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
os.makedirs("models/reranker", exist_ok=True)
shutil.copy(hf_hub_download(RR, "onnx/model_quint8_avx2.onnx"), "models/reranker/model_quint8_avx2.onnx")
shutil.copy(hf_hub_download(RR, "tokenizer.json"), "models/reranker/tokenizer.json")
!ls -la models/reranker

## Bộ câu hỏi eval
Nếu repo đã có `data/eval/questions.jsonl` (đã duyệt tay) thì dùng. Chưa có thì sinh ứng viên bằng LLM,
**kết quả với bộ chưa duyệt chỉ để tham khảo**.

In [ ]:
if not os.path.exists("data/eval/questions.jsonl"):
    !python -m vnlaw_rag.eval.make_questions --n 160 --out data/eval/candidates.jsonl
    shutil.copy("data/eval/candidates.jsonl", "data/eval/questions.jsonl")
    print("ĐANG DÙNG CÂU HỎI CHƯA DUYỆT")
!wc -l data/eval/questions.jsonl
!head -n 3 data/eval/questions.jsonl

## Ablation
Chạy với cả 2 embedding (bản sentence-transformers) và bản e5 int8 ONNX (bản thật sự deploy).

In [ ]:
for name, extra in [
    ("e5-small", "--dense-dir data/processed/dense_e5-small"),
    ("bkai", "--dense-dir data/processed/dense_bkai"),
    ("e5-int8-onnx", "--dense-dir data/processed/dense --dense-backend onnx --dense-model models/embedder"),
]:
    if name == "e5-int8-onnx":
        # embed_config của index deploy ghi model gốc; trỏ sang file int8
        cfg = json.load(open("data/processed/dense/embed_config.json"))
        cfg["onnx_file"] = "model.int8.onnx"
        json.dump(cfg, open("data/processed/dense/embed_config.json", "w"))
    !python -m vnlaw_rag.eval.run_eval {extra} \
        --reranker models/reranker --reranker-backend onnx --reranker-onnx-file model_quint8_avx2.onnx \
        --out results/retrieval_{name}.json

In [ ]:
for f in sorted(glob.glob("results/retrieval_*.md")):
    print("###", f)
    print(open(f, encoding="utf-8").read())

## Kết quả theo loại câu hỏi (bản deploy)

In [ ]:
import pandas as pd

r = json.load(open("results/retrieval_e5-int8-onnx.json", encoding="utf-8"))
rows = []
for cfg_name, v in r.items():
    for t, m in v["by_type"].items():
        rows.append({"config": cfg_name, "type": t, **{k: m[k] for k in ("recall@1", "recall@5", "mrr", "n")}})
pd.DataFrame(rows).pivot(index="config", columns="type", values="recall@5")

## Đẩy lên HF Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path="models", repo_id=HF_REPO, path_in_repo="models")
api.upload_folder(folder_path="data/processed", repo_id=HF_REPO, path_in_repo="processed",
                  allow_patterns=["chunks.jsonl", "dense/*"])
api.upload_folder(folder_path="results", repo_id=HF_REPO, path_in_repo="results")
print(f"https://huggingface.co/{HF_REPO}")